# MPR-Agent — graph-based multi-agent pipeline (ViNumQA)

Implementation of Nguyen et al., *"A Graph-Based Agent Approach to Numerical
Reasoning Question Answering"* ([VLSP 2025](https://aclanthology.org/2025.vlsp-1.29/)).
All logic lives in `agentic/` — see `README.md` beside this notebook for the
design and full ablation results. This notebook just runs it and prints PA/EA.

**Three ways to run this, only one applies at a time** — everything
environment-specific is marked as such below and stays commented out
otherwise, so switching just needs `MODEL` changed, no cells deleted:

| | `MODEL` | Needs | Environment-specific bits |
|---|---|---|---|
| **Local, Kaggle** | `Qwen3-4B` / `qwen3-4b-thinking` / `Gemma3-4B` | GPU + Internet | `!git clone` + `%cd` in the next cell — Settings -> Internet ON, Settings -> Accelerator = GPU. T4's 14.56 GB is tight: `LocalBackend` auto-retries at a smaller batch on OOM (may run slow on long prompts), see README.md. |
| **Local, Modal** | same three | A100-80GB Notebook | `!git clone` + `%cd` in the next cell, **Modal-labelled** variant — select GPU = A100-80GB in the Modal Notebook UI. No custom image needed (plain `transformers`, unlike the Unsloth/vLLM GRPO notebooks in `sft-grpo/`) — **untested on Modal specifically**, verify closely on first run. |
| **API** | any of the other eight (e.g. `DeepSeek-V4-Flash`) | `.env` with `API_KEY`/`BASE_URL` at the project root | None — runs on your own machine exactly like the DeepSeek-V4-Flash runs already done; leave both clone blocks commented |

Run the tests first either way: `pytest notebooks/vinumqa/graph-agent/tests -q`.

In [ ]:
# --- Kaggle only: clone the repo first, so .git/scorer.py/test.json/agentic/
# all come together with correct relative paths (uncomment both lines).
# Needs Internet ON in this notebook's Settings.
# -b chi: the OOM-retry fix in agentic/backends.py is only on this branch
# right now, not yet merged to main -- drop "-b chi" once it is.
# !git clone -b chi https://github.com/ntphuc149/NumReasoning4VietnameseFinancialText.git
# %cd NumReasoning4VietnameseFinancialText

# --- Modal only (A100-80GB Notebook): same idea, plain repo clone -- no
# custom image needed here (unlike sft-grpo/'s Unsloth+vLLM notebooks, see
# modal/README.md), plain `transformers` runs fine on Modal's default image.
# Untested on Modal specifically -- watch the first real run closely.
# Select GPU = A100-80GB in the Notebook UI before running this cell.
# !git clone -b chi https://github.com/ntphuc149/NumReasoning4VietnameseFinancialText.git
# %cd NumReasoning4VietnameseFinancialText

import sys
import time
from pathlib import Path

_here = Path.cwd()
ROOT = next((p for p in (_here, *_here.parents) if (p / ".git").exists()), None)
assert ROOT is not None, f"project root not found above {_here}"
HERE = ROOT / "notebooks" / "vinumqa" / "graph-agent"
sys.path.insert(0, str(HERE))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from agentic import AgentConfig, RunConfig, Runner, describe_backend

## Configuration

`MODEL` can be any of this repo's eleven baseline models — routed
automatically to local GPU or API (see `README.md`). `use_decomposition=False`
is the default: measured best on `DeepSeek-V4-Flash` (PA 0.7787, EA 0.8370,
full ablation table in `README.md`). Not yet re-verified on other models.

In [ ]:
MODEL = "qwen3-4b-thinking"
# MODEL = "Qwen3-4B" / "Gemma3-4B"   -- local, needs GPU
# MODEL = "DeepSeek-V4-Flash" / "gemma-3-27b-it" / "gemma-4-31B-it" / "gpt-oss-20b" /
#         "gpt-oss-120b" / "Llama-3.3-70B-Instruct" / "GLM-5.2" / "gpt-5-nano"  -- API, needs .env

agent_config = AgentConfig(
    model_subquery_gen=MODEL,
    model_subquery_ans=MODEL,
    model_planner=MODEL,
    model_fallback=MODEL,
    n_samples=15,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    prompt_lang="vi",
    vote_mode="canonical",
    use_prompt_ext=False,
    use_decomposition=False,   # best measured config on DeepSeek-V4-Flash -- NOT yet re-verified here
    max_workers_dataset=4,
    # rpm_limit=50, tpm_limit=100_000,   # GLM-5.2 on this endpoint -- see README.md
)

run_config = RunConfig(
    dataset_path="datasets/ViNumQA/origin/test.json",
    output_dir="notebooks/vinumqa/graph-agent/outputs",
    run_name=f"mpr-agent-{MODEL}",
    agent=agent_config,
    # limit=10,   # uncomment for a quick first check on a new MODEL/GPU combo before the full 497
)

runner = Runner(run_config)
print(f"MODEL={MODEL!r} -> {describe_backend(MODEL)} backend")

In [ ]:
started = time.time()
df = runner.run(show_progress=True)
scored, summary = runner.score(df)
runner.save(scored, summary)

print(f"\nPA: {summary['program_accuracy']}")
print(f"EA: {summary['execution_accuracy']}")
print(f"elapsed: {(time.time() - started) / 60:.1f} min")

### Error analysis

`oracle@n` (best of the 15 sampled candidates per sample) splits where PA is
lost: `oracle_pa − PA` is a correct program that voting picked wrong
(*heuristic selection error*, fixable by a better vote); `1 − oracle_pa` is a
sample where none of the 15 candidates were ever correct (*systematic
reasoning error* — voting cannot fix this, only a better model/prompt can).

In [ ]:
pa = summary["program_accuracy"]
oracle = summary["oracle_pa"]
print(f"correct and selected    : {pa}")
print(f"generated but out-voted : {oracle - pa}   (heuristic selection error)")
print(f"never generated         : {1 - oracle}   (systematic reasoning error)")
print(f"\nfallback rate           : {summary['fallback_rate']}")
print(f"empty predictions       : {summary['empty_rate']}")

wrong = scored[(scored["pa_score"] == 0) & scored["consensus"].notna()]
if len(wrong):
    print(f"\nmean consensus on PA-wrong samples: {wrong['consensus'].mean()}")
    print(f"of which unanimous (consensus 1.0): {(wrong['consensus'] == 1.0).mean()}")